In [11]:
from sklearn.pipeline import Pipeline
from sklearn import set_config

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeRegressor

from sklearn.model_selection import GridSearchCV

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

In [12]:
x_train, y_train = make_moons(n_samples=1000, noise=0.1, random_state=42)
x_train, X_test, y_train, y_test = train_test_split(x_train, y_train, test_size=0.2, random_state=42)

In [13]:
set_config(display="diagram")
preprocess = Pipeline(steps=[
        ("Median Imputation", SimpleImputer(strategy="median")),
        ("Standard Scaling", StandardScaler())
        ]
)

In [14]:
pipe = Pipeline(steps=[
    ("Preprocessing", preprocess),
    ("Decision Tree", DecisionTreeRegressor())
])

In [26]:
params_grid = {
    "Decision Tree__max_depth": [3, 5, 10, None],
    "Decision Tree__min_samples_split": [2, 5, 10],
    "Decision Tree__min_samples_leaf": [1, 2, 4],
    "Decision Tree__splitter": ["best", "random"],
    "Decision Tree__criterion": ["squared_error", "friedman_mse", "absolute_error"],
}

grid_search = GridSearchCV(pipe, params_grid, cv=5, n_jobs=1)
grid_search.fit(x_train, y_train)
print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation Score: {:.2f}".format(grid_search.best_score_))

Best Parameters: {'Decision Tree__criterion': 'friedman_mse', 'Decision Tree__max_depth': None, 'Decision Tree__min_samples_leaf': 1, 'Decision Tree__min_samples_split': 10, 'Decision Tree__splitter': 'random'}
Best Cross-Validation Score: 0.98


In [27]:
pipe = Pipeline(steps=[
    ("Preprocessing", preprocess),
    ("Decision Tree", DecisionTreeRegressor(criterion=grid_search.best_params_["Decision Tree__criterion"],
                                           max_depth=grid_search.best_params_["Decision Tree__max_depth"],
                                           min_samples_split=grid_search.best_params_["Decision Tree__min_samples_split"],
                                           min_samples_leaf=grid_search.best_params_["Decision Tree__min_samples_leaf"],
                                           splitter=grid_search.best_params_["Decision Tree__splitter"]))
])

pipe

,steps,"[('Preprocessing', ...), ('Decision Tree', ...)]"
,transform_input,None
,memory,None
,verbose,False
,steps,"[('Median Imputation', ...), ('Standard Scaling', ...)]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None


In [23]:
pipe.fit(x_train, y_train)
result = pipe.score(X_test, y_test)
print(f"Test Accuracy: {result:.2f}")

Test Accuracy: 1.00
